# Download synapse mesh and clefts

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AllenInstitute/connectomics_at_cosyne/blob/main/examples/Tutorial_Download_Synapse_Mesh.ipynb)



Example workflow for downloading the meshes around a set of synapses, along with their synaptic clefts.

## Import packages and initialize CAVEclient

In [ ]:
!uv pip install caveclient
!uv pip install cloud-volume

from google.colab import userdata
import caveclient
client = caveclient.CAVEclient('minnie65_public',
                               auth_token=userdata.get('CAVE_TOKEN'))

client.version = 1718 # materialization version at which this notebook was run

## Initialize cloud-volume

The meshes of the neuron segmentation (__meshes__) and the segmentation of the synaptic clefts (__clefts__) are in two separate data buckets. Defined here:

In [ ]:
import numpy as np
import pandas as pd
from cloudvolume import CloudVolume, Bbox

In [ ]:
# Mesh source: graphene layer from client
mesh_vol = CloudVolume(client.info.segmentation_source(),
                       progress=False,
                       use_https=True,
                       secrets=userdata.get('CAVE_TOKEN'),
                       mip=0)
print(mesh_vol.mip_resolution(0))

# Cleft source: precomputed segmentation layer on BOSSdb)
cleft_vol = CloudVolume("precomputed://https://bossdb-open-data.s3.amazonaws.com/iarpa_microns/minnie/minnie65/clefts-sharded",
                        progress=False,
                        use_https=True,
                        secrets=userdata.get('CAVE_TOKEN'),
                        mip=0)
print(cleft_vol.mip_resolution(0))

Note that the native resolution of both datasets is: `8 x 8 x 40 nm`, the downsampled imagery resolution at which imaging was run.

Any returned resolution that is greater than this is linearly upsampled.


## Get a set of synapses to download

In [ ]:
# example neuron, as of version 1718
example_root_id = 864691135572530981

# synapse query to get output synapses from our example cell
syn_df = client.materialize.synapse_query(post_ids=[example_root_id],
                                         desired_resolution=[8,8,40],
                                         )[['id','pre_pt_root_id','post_pt_root_id','ctr_pt_position','pre_pt_supervoxel_id','post_pt_supervoxel_id']]

syn_df

Note: if you already have the synapse `id` of interest (for example, from collecting the connectivity matrix at a different data version), you can materialize from the synapse table directly:  

In [ ]:
alternate_syn_df = client.materialize.tables.synapses_pni_2(id=syn_df.id).query(
    select_columns = ['id','pre_pt_root_id','post_pt_root_id','ctr_pt_position','pre_pt_supervoxel_id','post_pt_supervoxel_id'],
)
alternate_syn_df.head()

## Download cleft as points
In this example we will take the *first* synapse returned in the `syn_df` table. Consider how best to iterate across all synapses for your analysis

In [ ]:
# Select the first point
syn_id = syn_df.id.iloc[0]
ctr = syn_df.ctr_pt_position.iloc[0]

# Set a reasonable bounding box around the ctr_pt (here: 1 um in 8 x 8 x 40 units)
bounds = np.vstack([ctr - np.array([125, 125, 25]),
                    ctr + np.array([125, 125, 25]),] )

bounds

In [ ]:
# download
cleft = cleft_vol.download(bbox = Bbox(bounds[0], bounds[1]),
                           segids = [syn_id],
                           mip=0).squeeze()

Now turn the sparse tensor volume cutout object into a set of points for either meshing or visualization

In [ ]:
# Find where the cleft for the synapse id is
cleft_inds = np.where(cleft)

# calculate the voxel offset (given bounding box download)
min_inds = np.tile(bounds[0], (np.shape(cleft_inds)[1], 1) ).T

cleft_inds = min_inds+cleft_inds
cleft_inds

## Download meshes around synapse
Here we will use a handy feature of the PyChunkedGraph to download only the meshes around the *first* synapse of interest. We will load the local 'chunk' (up to agglomeration level `3`) of the segmented object.

This is a fast lookup, because we already have the `supervoxel` (agglomeration level `1`) from the synapse query.

*A note of caution* : because the 'chunks' of the chunked graph are not aligned to biology, you may find a larger level of agglomeration is necessary to see the local mesh features. Consider level `4`.

In [ ]:
%%time
supervoxel_ids = syn_df.iloc[0][['pre_pt_supervoxel_id','post_pt_supervoxel_id']].values

l3_ids = client.chunkedgraph.get_roots(supervoxel_ids, stop_layer=3)

all_meshes = mesh_vol.mesh.get(segids=l3_ids)
all_meshes

**Data resolution:** these meshes are downloaded at a converted `1 x 1 x 1 nm` resolution by default. In this example, I upscale the cleft download to match.

## Visualize with PyVista
Using pyvista, convert the meshes and cleft points to polydata, and overlay together

**Note:** unfortunately pyvista interactive mode does not work in google colab. I recommend doing this in local python rather than in the cloud

In [ ]:
!uv pip install pyvista[all]

In [ ]:
import pyvista as pv

def mesh_poly_from_draco(mesh):
    vertices = mesh.vertices
    faces = mesh.faces

    # add a column of all 3s to the faces
    padded_faces = np.concatenate([np.full((faces.shape[0], 1), 3), faces], axis=1)

    mesh_poly = pv.PolyData(vertices, faces=padded_faces)

    # Flip the y axis so pial surface is up
    mesh_poly.points[:, 1] *= -1

    return mesh_poly

def mesh_poly_from_cleft_points(points, resolution_conversion=np.array([1,1,1])):
    cleft_poly = pv.PolyData(cleft_inds.T)

    # Flip the y axis so pial surface is up
    cleft_poly.points[:, 1] *= -1

    # rescale to match mesh resolution
    cleft_poly.points = cleft_poly.points*resolution_conversion

    return cleft_poly

In [ ]:
cleft_inds

In [ ]:
pre_syn_poly = mesh_poly_from_draco(all_meshes[l3_ids[0]])
post_syn_poly = mesh_poly_from_draco(all_meshes[l3_ids[1]])

cleft_poly = mesh_poly_from_cleft_points(cleft_inds, np.array([8,8,40]))

In [ ]:
# Seems that only static plotting is supported by colab at the moment
pv.global_theme.jupyter_backend = 'static'
pv.global_theme.notebook = True
pv.start_xvfb()

plotter = pv.Plotter(image_scale=10)

plotter.add_mesh(pre_syn_poly, color=[.48, .67, .06], opacity=0.75)
plotter.add_mesh(post_syn_poly, color=[.06, .48, .67], opacity=0.75)

plotter.add_mesh(cleft_poly, point_size=6, color="orange", opacity=0.5)

plotter.show()